# Late Delivery Prediction: Data and Models

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg
from dotenv import load_dotenv
from scipy.stats import chi2_contingency, mannwhitneyu
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / '.env').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / '.env').is_file():
    raise FileNotFoundError('Could not locate .env. Start Jupyter from the project root or notebooks/.')

load_dotenv(PROJECT_ROOT / '.env')
required_settings = ('DB_HOST', 'DB_PORT', 'DB_NAME', 'DB_USER', 'DB_PASSWORD')
missing_settings = [name for name in required_settings if not os.getenv(name)]
if missing_settings:
    raise RuntimeError('Missing database settings: ' + ', '.join(missing_settings))

NUMERIC_FEATURES = ['promised_lead_time_days', 'item_count', 'total_order_value']
STATISTICAL_CATEGORICAL_FEATURES = ['purchase_month', 'purchase_weekday', 'customer_state', 'has_out_of_state_seller']
MODEL_CATEGORICAL_FEATURES = ['purchase_month', 'customer_state', 'has_out_of_state_seller']
STATISTICAL_FEATURES = NUMERIC_FEATURES + STATISTICAL_CATEGORICAL_FEATURES
MODEL_FEATURES = NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES


## Data and Splits

In [ ]:
FEATURE_QUERY = '''
SELECT
    order_id,
    purchase_date_key,
    purchase_month,
    purchase_weekday,
    promised_lead_time_days,
    customer_state,
    has_out_of_state_seller,
    item_count,
    total_order_value,
    is_ml_eligible,
    is_timeliness_eligible,
    is_late
FROM mart.order_delivery_ml_features
ORDER BY purchase_date_key, order_id
'''

with psycopg.connect(
    host=os.environ['DB_HOST'],
    port=int(os.environ['DB_PORT']),
    dbname=os.environ['DB_NAME'],
    user=os.environ['DB_USER'],
    password=os.environ['DB_PASSWORD'],
) as connection:
    features = pd.read_sql_query(FEATURE_QUERY, connection)

features['purchase_date'] = pd.to_datetime(
    features['purchase_date_key'].astype(str), format='%Y%m%d', errors='raise'
)
for column in ('is_ml_eligible', 'is_timeliness_eligible', 'is_late'):
    features[column] = features[column].astype('boolean')
features['has_out_of_state_seller'] = features['has_out_of_state_seller'].map(
    {True: 'yes', False: 'no'}
).astype(object)

labelled = features.loc[
    features['is_ml_eligible']
    & features['is_timeliness_eligible']
    & features['is_late'].notna()
].copy()
if labelled.empty:
    raise ValueError('No eligible labelled rows. Check the ML feature mart pipeline.')

ordered = labelled.sort_values(['purchase_date', 'order_id']).reset_index(drop=True)
train_end = int(len(ordered) * 0.70)
validation_end = int(len(ordered) * 0.85)
train = ordered.iloc[:train_end].copy()
validation = ordered.iloc[train_end:validation_end].copy()
test = ordered.iloc[validation_end:].copy()

def split_row(name, frame):
    return {
        'split': name,
        'orders': len(frame),
        'first_purchase': frame['purchase_date'].min(),
        'last_purchase': frame['purchase_date'].max(),
        'late_rate': frame['is_late'].mean(),
    }

population_summary = pd.DataFrame({
    'orders_in_mart': [len(features)],
    'eligible_labelled_orders': [len(labelled)],
    'late_rate': [labelled['is_late'].mean()],
})
split_summary = pd.DataFrame([
    split_row('train', train),
    split_row('validation', validation),
    split_row('test', test),
])
display(population_summary, split_summary)


## Training-Set Checks

In [ ]:
def outlier_summary(frame, columns):
    rows = []
    for column in columns:
        values = pd.to_numeric(frame[column], errors='coerce').dropna()
        q1, q3 = values.quantile([0.25, 0.75])
        lower, upper = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
        rows.append({
            'feature': column,
            'min': values.min(),
            'median': values.median(),
            'max': values.max(),
            'outside_iqr': int(((values < lower) | (values > upper)).sum()),
        })
    return pd.DataFrame(rows).set_index('feature')

def calculate_vif(frame, columns):
    numeric = frame[columns].apply(pd.to_numeric, errors='coerce')
    numeric = numeric.fillna(numeric.median())
    rows = []
    for column in columns:
        target = numeric[column].to_numpy()
        others = numeric.drop(columns=column).to_numpy()
        total_sum_squares = np.square(target - target.mean()).sum()
        if np.isclose(total_sum_squares, 0):
            vif = np.inf
        else:
            design = np.column_stack([np.ones(len(target)), others])
            coefficients, *_ = np.linalg.lstsq(design, target, rcond=None)
            residual_sum_squares = np.square(target - design @ coefficients).sum()
            r_squared = 1 - residual_sum_squares / total_sum_squares
            vif = np.inf if 1 - r_squared < 1e-10 else 1 / (1 - r_squared)
        rows.append({'feature': column, 'vif': vif})
    return pd.DataFrame(rows).sort_values('vif', ascending=False)

def grouped_categories(series, minimum_count=30):
    values = series.astype('string').fillna('<missing>')
    rare = values.value_counts()[lambda counts: counts < minimum_count].index
    return values.where(~values.isin(rare), '<other>')

missing_profile = train[STATISTICAL_FEATURES].isna().agg(['sum', 'mean']).T.rename(
    columns={'sum': 'missing_rows', 'mean': 'missing_rate'}
)
numeric_by_label = train.groupby('is_late', observed=True)[NUMERIC_FEATURES].agg(['count', 'median', 'mean']).T

mann_whitney_rows = []
for column in NUMERIC_FEATURES:
    late_values = pd.to_numeric(train.loc[train['is_late'].eq(True), column], errors='coerce').dropna()
    non_late_values = pd.to_numeric(train.loc[train['is_late'].eq(False), column], errors='coerce').dropna()
    statistic, p_value = mannwhitneyu(late_values, non_late_values, alternative='two-sided')
    mann_whitney_rows.append({
        'feature': column,
        'late_median': late_values.median(),
        'non_late_median': non_late_values.median(),
        'p_value': p_value,
    })
mann_whitney_results = pd.DataFrame(mann_whitney_rows).set_index('feature').sort_values('p_value')

chi_square_rows = []
for column in STATISTICAL_CATEGORICAL_FEATURES:
    contingency = pd.crosstab(grouped_categories(train[column]), train['is_late'])
    chi_square, p_value, degrees_of_freedom, expected = chi2_contingency(contingency)
    phi_squared = chi_square / expected.sum()
    denominator = min(contingency.shape[0] - 1, contingency.shape[1] - 1)
    chi_square_rows.append({
        'feature': column,
        'p_value': p_value,
        'cramers_v': np.sqrt(phi_squared / denominator) if denominator else np.nan,
        'minimum_expected_count': expected.min(),
    })
chi_square_results = pd.DataFrame(chi_square_rows).set_index('feature').sort_values('p_value')

display(missing_profile, numeric_by_label, outlier_summary(train, NUMERIC_FEATURES))
display(calculate_vif(train, NUMERIC_FEATURES), mann_whitney_results, chi_square_results)


## Training and Evaluation

In [ ]:
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler()),
    ]), NUMERIC_FEATURES),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('one_hot', OneHotEncoder(handle_unknown='ignore')),
    ]), MODEL_CATEGORICAL_FEATURES),
])

X_train, y_train = train[MODEL_FEATURES], train['is_late'].astype(int)
X_validation, y_validation = validation[MODEL_FEATURES], validation['is_late'].astype(int)
X_test, y_test = test[MODEL_FEATURES], test['is_late'].astype(int)

def evaluate_probabilities(y_true, probabilities, threshold=None):
    top_count = max(1, int(np.ceil(len(probabilities) * 0.10)))
    top_indices = np.argsort(probabilities)[-top_count:]
    result = {
        'pr_auc': average_precision_score(y_true, probabilities),
        'roc_auc': roc_auc_score(y_true, probabilities),
        'brier_score': brier_score_loss(y_true, probabilities),
        'recall_at_top_10_pct': y_true.iloc[top_indices].sum() / y_true.sum(),
    }
    if threshold is not None:
        flagged = probabilities >= threshold
        result['flagged_orders'] = int(flagged.sum())
        result['precision_at_threshold'] = y_true[flagged].mean() if flagged.any() else np.nan
        result['recall_at_threshold'] = y_true[flagged].sum() / y_true.sum()
    return result

logistic_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1_000, random_state=42)),
])
logistic_pipeline.fit(X_train, y_train)
logistic_probabilities = logistic_pipeline.predict_proba(X_validation)[:, 1]

validation_results = pd.DataFrame([
    {'model': 'Logistic regression', **evaluate_probabilities(y_validation, logistic_probabilities)},
]).set_index('model')
display(validation_results)


In [ ]:
ALERT_FRACTION = 0.10

validation_probabilities = logistic_pipeline.predict_proba(X_validation)[:, 1]
selected_threshold = float(np.quantile(validation_probabilities, 1 - ALERT_FRACTION))
test_probabilities = logistic_pipeline.predict_proba(X_test)[:, 1]

final_results = pd.DataFrame([
    {'split': 'validation', **evaluate_probabilities(y_validation, validation_probabilities, selected_threshold)},
    {'split': 'test', **evaluate_probabilities(y_test, test_probabilities, selected_threshold)},
]).set_index('split')

feature_names = logistic_pipeline.named_steps['preprocessor'].get_feature_names_out()
model = logistic_pipeline.named_steps['model']
feature_effects = pd.DataFrame({
    'feature': feature_names,
    'coefficient': model.coef_[0],
})
feature_effects['absolute_effect'] = feature_effects['coefficient'].abs()
feature_effects = feature_effects.sort_values('absolute_effect', ascending=False).drop(columns='absolute_effect')

display(final_results, feature_effects)
